# Multi-Position Trader Engine

This notebook is a cleaned public version of the multi-position trader loop I built while researching prediction-market reactions to underlying market moves.

It assumes an upstream notebook has already created a `trade_table`: one row per timestamp and outcome side, with prediction-market bid/ask levels, an underlying-market signal, staleness checks, and a proposed trade size. I do not include the original market data or market identifiers here.


In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))


## Expected Table

The engine expects a prepared `trade_table`. The important design choice is that Yes and No are separate rows, so entries and exits must always check the row's `outcome` before using that row's bid or ask.


In [ ]:
REQUIRED_COLUMNS = [
    "timestamp_utc",
    "outcome",
    "ask_price_1",
    "bid_price_1",
    "trade_size",
    "poly_stale_mins",
    "rolling_deviation",
    "liquid_book",
]

OPTIONAL_BOOK_COLUMNS = [
    "ask_price_2", "ask_price_3",
    "bid_price_2", "bid_price_3",
    "ask_size_1", "ask_size_2", "ask_size_3",
    "bid_size_1", "bid_size_2", "bid_size_3",
]

if "trade_table" in globals():
    missing_cols = [col for col in REQUIRED_COLUMNS if col not in trade_table.columns]
    if missing_cols:
        raise ValueError(f"trade_table is missing columns: {missing_cols}")
    trade_table = trade_table.sort_values(["timestamp_utc", "outcome"]).reset_index(drop=True)
    display(trade_table.head())
else:
    print("Load or create trade_table before running the backtest cells.")
    print("Required columns:")
    print(REQUIRED_COLUMNS)


## Parameters

These are not fitted parameters. They are readable defaults for testing the engine mechanics: stale-price filters, holding time, stop loss, take profit, and entry thresholds.


In [ ]:
Z_SCORE_1 = 1.96
MAX_STALE_MINS = 8
MAX_HOLD_TIME = 60
STOP_LOSS = 0.04
TAKE_PROFIT = 0.20
STOP_LOSS_ACTIVATION_MINUTES = 10
THRESHOLD_PRICE = 0.99
ALLOW_OVERNIGHT_RETURNS = False

VALID_HOLDING_TIME_COL = "valid_holding_time"
VALID_HISTORY_COL = "valid_history"


## Small Helpers

I keep these functions boring on purpose. One function calculates elapsed minutes, and one function chooses the executable price for a buy or sell from the current row.


In [ ]:
def minutes_between(earlier_time, later_time):
    return (later_time - earlier_time).total_seconds() / 60


def execution_price(row, action):
    if action == "Buy":
        price_candidate = row["ask_price_1"]
        if pd.notna(price_candidate):
            return price_candidate
        return np.nan

    if action == "Sell":
        price_candidate = row["bid_price_1"]
        if pd.notna(price_candidate):
            return price_candidate
        return np.nan

    return np.nan


## Multi-State Entry

The trader state is a dictionary with a list of open positions. Each new position is just another dictionary appended to that list.


In [ ]:
def multi_state_reset():
    multi_state = {
        "positions": []
    }

    return multi_state


def enter_multi_positions(row, multi_state, entry_side):
    price_candidate = execution_price(row, "Buy")

    if pd.notna(price_candidate):
        if entry_side == "Yes":
            position = {
                "position_yes": 1,
                "entry_timestamp": row["timestamp_utc"],
                "entry_price": price_candidate,
                "entry_side": "Yes",
                "trade_size": row["trade_size"],
                "total_entry_cost": price_candidate * row["trade_size"],
                "liquidation_value": row["bid_price_1"],
                "stop_loss_price": row["bid_price_1"] - STOP_LOSS,
                "take_profit_price": price_candidate + TAKE_PROFIT,
                "position_number": len(multi_state["positions"]),
            }

        if entry_side == "No":
            position = {
                "position_yes": -1,
                "entry_timestamp": row["timestamp_utc"],
                "entry_price": price_candidate,
                "entry_side": "No",
                "trade_size": row["trade_size"],
                "total_entry_cost": price_candidate * row["trade_size"],
                "liquidation_value": row["bid_price_1"],
                "stop_loss_price": row["bid_price_1"] - STOP_LOSS,
                "take_profit_price": price_candidate + TAKE_PROFIT,
                "position_number": len(multi_state["positions"]),
            }

        multi_state["positions"].append(position)
        return multi_state

    return None


## Entry Signal

This function decides whether the current row wants a Yes or No entry. It checks the row itself, not the current position state. That separation matters because the multi-position engine can decide separately whether to take every valid signal or restrict entries.


In [ ]:
def _valid_window(row, col_name):
    if col_name not in row.index:
        return True
    return bool(row[col_name]) or ALLOW_OVERNIGHT_RETURNS


def multi_should_enter(row):
    price_candidate = execution_price(row, "Buy")

    if pd.isna(price_candidate):
        return None

    base_condition = (
        row["trade_size"] > 0
        and row["poly_stale_mins"] <= MAX_STALE_MINS
        and _valid_window(row, VALID_HOLDING_TIME_COL)
        and _valid_window(row, VALID_HISTORY_COL)
        and price_candidate <= THRESHOLD_PRICE
        and pd.notna(row[["ask_price_1", "bid_price_1"]]).all()
        and bool(row["liquid_book"])
    )

    if not base_condition:
        return None

    if row["rolling_deviation"] >= Z_SCORE_1:
        trade_side = "Yes"
        if row["outcome"] == trade_side:
            return trade_side
        return None

    if row["rolling_deviation"] <= -Z_SCORE_1:
        trade_side = "No"
        if row["outcome"] == trade_side:
            return trade_side
        return None

    return None


## Exit Logic

The exit function only evaluates a position on rows with the same outcome side. This prevents a No position from being closed using a Yes bid, or the reverse.


In [ ]:
def multi_state_should_exit(row, position):
    if row["outcome"] != position["entry_side"]:
        return None

    price_candidate = execution_price(row, "Sell")
    time_elapsed = minutes_between(position["entry_timestamp"], row["timestamp_utc"])

    if pd.notna(price_candidate):
        if time_elapsed >= STOP_LOSS_ACTIVATION_MINUTES:
            if price_candidate <= position["stop_loss_price"]:
                return "stop_loss_hit"

        if price_candidate >= position["take_profit_price"]:
            return "take_profit_hit"

        if time_elapsed >= MAX_HOLD_TIME:
            return "max_time_hit"

    return None


def multi_state_exit_position(row, position, exit_reason):
    price_candidate = execution_price(row, "Sell")

    if pd.notna(price_candidate):
        trade_ledger = {
            "entry_time": position["entry_timestamp"],
            "exit_time": row["timestamp_utc"],
            "entry_price": position["entry_price"],
            "trade_size": position["trade_size"],
            "entry_cost": position["total_entry_cost"],
            "exit_price": price_candidate,
            "exit_reason": exit_reason,
            "time_held": minutes_between(position["entry_timestamp"], row["timestamp_utc"]),
            "entry_side": position["entry_side"],
            "pnl": (price_candidate - position["entry_price"]) * position["trade_size"],
            "liquidation_price": position["liquidation_value"],
        }
        return trade_ledger

    return None


## Backtest Loop

This is the core loop. On every row, it first checks exits for all open positions, carries forward survivors, force-closes positions on the last available row for their own outcome side, and then opens a new position if the row has a valid entry signal.


In [ ]:
def _last_outcome_indices(trade_table):
    return (
        trade_table[trade_table["outcome"] == "Yes"].index[-1],
        trade_table[trade_table["outcome"] == "No"].index[-1],
    )


def run_multi_backtest(trade_table, debug=True):
    multi_state = multi_state_reset()
    multi_trades_ledger = []
    debug_rows = []
    last_outcome_indices = _last_outcome_indices(trade_table)

    for idx, row in trade_table.iterrows():
        still_open_positions = []
        exit_reasons = []
        entry_side = multi_should_enter(row)
        entry_bool = entry_side is not None and idx not in last_outcome_indices
        yes_entry_bool = entry_side == "Yes" and idx not in last_outcome_indices
        no_entry_bool = entry_side == "No" and idx not in last_outcome_indices

        for position in multi_state["positions"]:
            exit_reason = multi_state_should_exit(row, position)

            if exit_reason is not None:
                trade_ledger = multi_state_exit_position(row, position, exit_reason)
                multi_trades_ledger.append(trade_ledger)
                exit_reasons.append(exit_reason)
            else:
                still_open_positions.append(position)

        multi_state["positions"] = still_open_positions

        if idx in last_outcome_indices and len(multi_state["positions"]) > 0:
            last_open_positions = []

            for position in multi_state["positions"]:
                if row["outcome"] == position["entry_side"]:
                    exit_reasons.append("end_of_day")

                    trade_ledger = {
                        "entry_time": position["entry_timestamp"],
                        "exit_time": row["timestamp_utc"],
                        "entry_price": position["entry_price"],
                        "trade_size": position["trade_size"],
                        "entry_cost": position["total_entry_cost"],
                        "exit_price": row["bid_price_1"],
                        "exit_reason": "end_of_day",
                        "time_held": minutes_between(position["entry_timestamp"], row["timestamp_utc"]),
                        "entry_side": position["entry_side"],
                        "pnl": (row["bid_price_1"] - position["entry_price"]) * position["trade_size"],
                        "liquidation_price": position["liquidation_value"],
                    }

                    multi_trades_ledger.append(trade_ledger)
                else:
                    last_open_positions.append(position)

            multi_state["positions"] = last_open_positions

        if entry_side is not None and idx not in last_outcome_indices:
            multi_state = enter_multi_positions(row, multi_state, entry_side)

        if debug:
            debug_row = {
                "timestamp_utc": row["timestamp_utc"],
                "outcome": row["outcome"],
                "exit_reasons": ",".join(exit_reasons),
                "num_exits_this_row": len(exit_reasons),
                "entry_signal": entry_side,
                "entry_bool": entry_bool,
                "yes_entry_bool": yes_entry_bool,
                "no_entry_bool": no_entry_bool,
                "trade_size": row["trade_size"],
            }

            debug_rows.append(debug_row)

    multi_trades_df = pd.DataFrame(multi_trades_ledger)
    multi_debug_df = pd.DataFrame(debug_rows)

    if not debug:
        return multi_trades_df

    return multi_trades_df, multi_debug_df


## Summaries

These summary functions keep raw signal counts separate from closed-trade P&L. That makes it easier to see whether the trader skipped signals, left positions open, or force-closed positions at the end of the available window.


In [ ]:
def multi_trade_insights(multi_trades_df):
    multi_summary = {
        "trades": 0,
        "total_pnl": 0,
        "average_pnl": np.nan,
        "win_rate": np.nan,
        "best_trade": np.nan,
        "worst_trade": np.nan,
        "pct_side_yes": np.nan,
        "total_turnover": 0,
    }

    if len(multi_trades_df) > 0:
        multi_summary["trades"] = len(multi_trades_df)
        multi_summary["total_pnl"] = multi_trades_df["pnl"].sum()
        multi_summary["average_pnl"] = multi_trades_df["pnl"].mean()
        multi_summary["win_rate"] = (multi_trades_df["pnl"] > 0).mean()
        multi_summary["best_trade"] = multi_trades_df["pnl"].max()
        multi_summary["worst_trade"] = multi_trades_df["pnl"].min()
        multi_summary["pct_side_yes"] = (multi_trades_df["entry_side"] == "Yes").mean()
        multi_summary["total_turnover"] = multi_trades_df["trade_size"].sum()

    multi_summary_df = pd.DataFrame([multi_summary])
    return multi_summary_df


def multi_debug_summary(multi_debug_df):
    multi_debug_summary = {
        "yes_entry_count": 0,
        "no_entry_count": 0,
        "total_entry_count": 0,
    }

    if len(multi_debug_df) > 0:
        multi_debug_summary["yes_entry_count"] = multi_debug_df["yes_entry_bool"].sum()
        multi_debug_summary["no_entry_count"] = multi_debug_df["no_entry_bool"].sum()
        multi_debug_summary["total_entry_count"] = multi_debug_df["entry_bool"].sum()

    multi_debug_summary_df = pd.DataFrame([multi_debug_summary])
    return multi_debug_summary_df


## Run The Engine

Run this cell after loading a prepared `trade_table` from an upstream data notebook.


In [ ]:
if "trade_table" in globals():
    multi_trades_df, multi_debug_df = run_multi_backtest(trade_table, debug=True)
    multi_summary_df = multi_trade_insights(multi_trades_df)
    multi_debug_summary_df = multi_debug_summary(multi_debug_df)

    display(multi_debug_summary_df)
    display(multi_summary_df)
    display(multi_trades_df.head())
else:
    print("trade_table is not loaded in this notebook session.")


## Next Execution Upgrade

The next version replaces single-level execution with an order-book fill estimator. The entry function would use displayed ask levels to estimate average buy price and filled size; the exit function would do the same with bid levels. The maximum acceptable spread should be row-specific, starting from a conservative base and widening only when the signal and displayed liquidity justify it.
